# 🏆 INDEX-MATCH Maze — Solution Walkthrough

**Category:** Formulas | **Difficulty:** 🔵 Journeyman

This notebook walks through the INDEX-MATCH challenge using Python and `pandas`.
We demonstrate left-lookups, two-dimensional lookups, and dynamic references.

> In Excel you would use `=INDEX(...)` and `=MATCH(...)` formulas directly. Here we replicate the logic in Python.

## Prerequisites

```bash
pip install pandas openpyxl
```

In [ ]:
import pandas as pd
import numpy as np

print('Libraries loaded!')

## Step 1 — Create the Labyrinth Data

Our maze is an employee table where the **Employee ID** column is in the *middle*,
not the leftmost column — which makes VLOOKUP useless here.

In [ ]:
# Notice: EmployeeID is NOT the first column — VLOOKUP can't look left!
employees = pd.DataFrame({
    'Department': ['Engineering', 'Marketing', 'Finance', 'HR', 'Engineering', 'Marketing'],
    'Name':       ['Alice', 'Bob', 'Carol', 'Dave', 'Eve', 'Frank'],
    'EmployeeID': ['E101', 'E102', 'E103', 'E104', 'E105', 'E106'],
    'Salary':     [90000, 75000, 82000, 68000, 95000, 71000],
    'YearsExp':   [5, 3, 7, 2, 8, 4]
})

# Sales grid for two-dimensional lookup
sales_grid = pd.DataFrame({
    'Region':  ['North', 'South', 'East', 'West'],
    'Q1': [120000, 98000, 145000, 110000],
    'Q2': [132000, 105000, 138000, 120000],
    'Q3': [115000, 112000, 152000, 108000],
    'Q4': [140000, 99000, 160000, 125000]
})

print('Employee Table (EmployeeID is NOT the first column):')
print(employees.to_string(index=False))
print()
print('Sales Grid:')
print(sales_grid.to_string(index=False))

## Step 2 — Left-Lookup (INDEX-MATCH)

**Challenge:** Given an `EmployeeID`, return the employee's `Name` — which is to the *LEFT* of `EmployeeID`.

**Excel formula:**
```
=INDEX(A2:A7, MATCH("E103", C2:C7, 0))
```

- `MATCH("E103", C2:C7, 0)` → returns position 3 (row index of E103)
- `INDEX(A2:A7, 3)` → returns the Name in row 3 → "Carol"

In [ ]:
def index_match(return_col, search_col, lookup_value):
    """Equivalent to =INDEX(return_col, MATCH(lookup_value, search_col, 0))"""
    try:
        # MATCH — find the position (0-based index)
        position = search_col.tolist().index(lookup_value)
        # INDEX — retrieve the value at that position in the return column
        return return_col.iloc[position]
    except ValueError:
        return '#N/A'

# Left-lookup: find Name given EmployeeID
target_id = 'E103'
name = index_match(employees['Name'], employees['EmployeeID'], target_id)
dept = index_match(employees['Department'], employees['EmployeeID'], target_id)

print(f'Looking up EmployeeID: {target_id}')
print(f'  Name       : {name}')
print(f'  Department : {dept}')

## Step 3 — Two-Dimensional Lookup

**Challenge:** Find the sales figure for a specific Region AND Quarter.

**Excel formula:**
```
=INDEX(B2:E5, MATCH("East", A2:A5, 0), MATCH("Q3", B1:E1, 0))
```
- First MATCH → row position for "East"
- Second MATCH → column position for "Q3"
- INDEX → value at that row/column intersection

In [ ]:
def two_dim_lookup(df, row_key_col, row_value, col_value):
    """Two-dimensional INDEX-MATCH lookup."""
    # MATCH for row
    row_mask = df[row_key_col] == row_value
    if not row_mask.any():
        return '#N/A'
    # MATCH for column
    if col_value not in df.columns:
        return '#N/A'
    # INDEX — retrieve the intersecting value
    return df.loc[row_mask, col_value].values[0]

# Find Q3 sales for the East region
region = 'East'
quarter = 'Q3'
value = two_dim_lookup(sales_grid, 'Region', region, quarter)

print(f'Sales for Region={region}, Quarter={quarter}: ${value:,.0f}')

## Step 4 — Dynamic Reference (all regions for one quarter)

Because INDEX-MATCH is flexible, we can easily loop over multiple lookups
— equivalent to filling a formula down a column in Excel.

In [ ]:
# Retrieve Q4 sales for all regions dynamically
target_quarter = 'Q4'

print(f'Dynamic lookup — {target_quarter} Sales by Region:')
print('-' * 30)
for region in sales_grid['Region']:
    val = two_dim_lookup(sales_grid, 'Region', region, target_quarter)
    print(f'  {region:<8} : ${val:>10,.0f}')

## ✅ Challenge Complete!

You've successfully:
- ✅ Performed a **left-lookup** using INDEX-MATCH (retrieved Name from EmployeeID)
- ✅ Performed a **two-dimensional lookup** (Region × Quarter → Sales)
- ✅ Built a **dynamic reference** that loops across all rows

### Key Takeaways
- `MATCH` finds the *position* of a value in a range
- `INDEX` retrieves the value at a given position
- Together they are more flexible than VLOOKUP because:
  - The lookup column can be *anywhere* (not just the first column)
  - You can do two-dimensional lookups

### 🔗 Try Next:
- **[Pivot Pyramid](../../pivot-tables/pivot-pyramid/)** — Summarize data with pivot tables
- **[Dashboard Duel](../../data-analysis/dashboard-duel/)** — Visualize your data